# TalentLens - candidate ranking demo

Ranks candidates for a role on **meaning, not keywords**, then discounts by **credibility**
(kills keyword-stuffers) and **availability** (down-weights the unreachable):

```
fit = (0.55·semantic + 0.30·skill_coverage + 0.15·seniority) × credibility × availability
```

The role is **config, not code**: `jobs/senior_ai_engineer/description.md` (the full JD,
embedded as intent) + `spec.json` (skill families, weights, credibility rules). This notebook
clones the repo, pulls the candidate file, and runs the same `rank.py` you'd run locally.

**Fast path:** candidate embeddings are *job-independent*, so they're precomputed once and
published. This notebook downloads them and only encodes the one-line job intent at run time -
the full 100K pool ranks in **~2-3 minutes on a plain CPU runtime** (no GPU needed). Just
`Runtime → Run all`. (Turn `USE_PRECOMPUTED` off to encode from scratch instead - then a GPU
runtime is recommended.)

In [ ]:
#@title Configuration  { display-mode: "form" }
REPO_URL = "https://github.com/anjalilaishram/talentlens.git"  #@param {type:"string"}
JOB = "jobs/senior_ai_engineer"  #@param {type:"string"}
#@markdown Rank the full pool? If off, uses the bundled 50-candidate sample for an instant run.
FULL_POOL = True  #@param {type:"boolean"}
#@markdown Direct Google Drive **file** link to candidates.jsonl (shared "anyone with the link").
DRIVE_FILE_URL = "https://drive.google.com/file/d/1FacTQ_l4jq4qEa2X_rim15tlrrwwLam2/view"  #@param {type:"string"}
#@markdown Use precomputed candidate embeddings (fast, no GPU). Turn off to encode from scratch.
USE_PRECOMPUTED = True  #@param {type:"boolean"}
#@markdown Direct Google Drive **file** link to the precomputed embeddings (.npz).
EMB_FILE_URL = "https://drive.google.com/file/d/1LFf25qkwjTvFayQNpAktFhrwtRKMvAaq/view"  #@param {type:"string"}
TOPK = 100  #@param {type:"integer"}
print(f"repo={REPO_URL}\njob={JOB} | full_pool={FULL_POOL} | precomputed={USE_PRECOMPUTED} | topk={TOPK}")

### 1. Get the code

In [ ]:
import os
os.chdir('/content')
!rm -rf /content/talentlens
!git clone --depth 1 $REPO_URL /content/talentlens
%cd /content/talentlens
!pip install -q -r requirements.txt

### 2. Get the candidate file

In [ ]:
EMB = ''
if FULL_POOL:
    !pip install -q gdown
    import gdown
    CANDIDATES = '/content/candidates.jsonl'
    gdown.download(url=DRIVE_FILE_URL.strip(), output=CANDIDATES, quiet=False, fuzzy=True)
    if USE_PRECOMPUTED:
        EMB = '/content/embeddings.npz'
        gdown.download(url=EMB_FILE_URL.strip(), output=EMB, quiet=False, fuzzy=True)
else:
    CANDIDATES = 'sample_candidates.json'  # tiny sample encodes instantly; no precompute needed
print('Candidates:', CANDIDATES, '| embeddings:', EMB or '(encode at run time)')

### 3. Rank

In [ ]:
import time
t0 = time.time()
EMB_ARG = f'--emb "{EMB}"' if EMB else ''
!python rank.py --job "$JOB" --candidates "$CANDIDATES" --out submission.csv --topk $TOPK $EMB_ARG
print(f"\nDone in {time.time()-t0:.0f}s -> submission.csv")

### 4. Results

`submission.csv` is the ranking: `candidate_id, rank, score, reasoning`. The bar chart shows
the 0-1 fit of the top candidates; the table shows the human-readable reasoning per pick.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.max_colwidth', 220)
df = pd.read_csv('submission.csv')
top = df.head(15)
plt.figure(figsize=(9, 0.4*len(top)+1))
plt.barh(top['candidate_id'][::-1], top['score'][::-1], color='#1f9d8f')
plt.xlabel('fit score (0-1)'); plt.title('Top candidates by fit'); plt.xlim(0, 1.05)
plt.tight_layout(); plt.show()
df.head(15)

In [ ]:
# Download the ranking (optional)
from google.colab import files
files.download('submission.csv')